# C2 · apertura — notebook de análisis (`debug`)

**Objeto:** ROXs12b  |  **Run:** `ROXs12b_realigned`  |  **Spec:** [`docs/spec_C2_codex_aperture_extraction.md`](../../../docs/spec_C2_codex_aperture_extraction.md)

Este notebook **no llama a la cadena**: rehace la extracción por apertura aquí dentro, con el código a la vista, para que puedas **probar, cambiar y ajustar sin tocar `musepipe`**. El notebook de auditoría equivalente es [`../C2_aperture.ipynb`](../C2_aperture.ipynb), que sí llama a la etapa.

Cómo está montado, y por qué:

1. **Perillas** arriba del todo, con el valor que usa la cadena para este run.
2. **Las funciones numéricas, copiadas literalmente** de `musepipe`. Se copian (en vez de importarse) para que puedas editarlas: todo lo que viene después usa estos nombres locales.
3. **Chequeo de deriva** — avisa si `musepipe` cambió y esta copia se quedó atrás.
4. El proceso **paso a paso**, cada uno con su diagnóstico.
5. **Comparación con el producto de la cadena**: con las perillas por defecto debe salir *idéntico*; en cuanto cambias algo, te dice qué se movió y dónde.

> Lo que NO se copia: `evaluate_psf_model` (es de C1, no es lo que se ajusta aquí) y el ensamblado del `SpectrumProduct`, que va como código plano más abajo.


In [ ]:
import json, sys
from pathlib import Path

import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

_here = Path.cwd()
ROOT = next(p for p in (_here, *_here.parents) if (p / 'musepipe').is_dir())
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'notebooks'))
import _nbcommon as nb

RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
RD = nb.run_dir(RUN_ID); SD = RD / 'stages'
CFG = json.loads((RD / 'config' / 'config.json').read_text(encoding='utf-8'))['config']
# El objeto se DERIVA del run (cadena declarada en su config), no se
# escribe: un literal aquí haría que un objeto nuevo heredase el nombre
# del primero, que es lo que vigila tests/test_no_hardcoded_target.py.
TARGET = nb.run_target(RUN_ID) or nb.display_name(RUN_ID)
print('objeto :', TARGET, '·', nb.display_name(RUN_ID))
print('run    :', RUN_ID)
print('stages :', SD)


## 1 · Perillas

Salen del **config resuelto de la etapa**, no del `config.json` crudo: C2 rellena defaults que no están escritos en el run, y copiarlos a mano es exactamente cómo se consigue un notebook que no reproduce la cadena. Cambia lo que quieras **debajo** de la lectura y vuelve a ejecutar: la comparación del final dirá qué efecto tuvo.


In [ ]:
from musepipe.stages.stage_x01_aperture import stage_x01_config_from_run

# `project_root=ROOT` no es opcional: musepipe resuelve rutas contra el cwd, y
# el cwd de un notebook es su propia carpeta, no la raíz del repo.
X01 = stage_x01_config_from_run(RUN_ID, project_root=ROOT)   # run + defaults de la etapa
APERTURE            = {'kind': 'box', 'size': 3}   # la caja que se compara con la cadena
APCORR_MODE         = X01.get('x01_aperture_correction', 'auto')
WINGS_INTACT        = bool(X01.get('x01_wings_intact_apcorr', True))
ANNULUS_BKG_PX      = X01.get('x01_annulus_bkg_px', [8.0, 14.0, 30.0])
N_CONTROLS          = int(X01.get('x01_control_apertures', 8))
EXCLUDE_ANGLE_DEG   = float(X01.get('x01_control_exclude_angle_deg', 25.0))
ERROR_MODE          = X01.get('x01_error_mode', 'auto')
BAD_WINDOWS_A       = X01.get('x01_bad_windows_A', [])
SKYLINE_WINDOWS_A   = X01.get('x01_skyline_windows_A', [])
INTERPOLATED_WIN_A  = X01.get('x01_interpolated_windows_A', [])

# ---- a partir de aquí, cambia lo que quieras probar ----

for _k, _v in {'apertura': APERTURE, 'apcorr': APCORR_MODE, 'wings-intact': WINGS_INTACT,
               'anillo fondo': ANNULUS_BKG_PX, 'controles': N_CONTROLS,
               'modo error': ERROR_MODE}.items():
    print(f'  {_k:14s} {_v}')


## 2 · Entradas

Las mismas que toma C2, y **de dónde sale cada una**. Ojo al cubo: cuando la corrección de apertura está activa, C2 **no** extrae del residual de 04b sino del cubo crudo de B2 con un fondo de anillo (*wings-intact*), porque el residual de 04b se come las alas del compañero y rompería la consistencia box3/box5 de la curva de crecimiento. Esa decisión se replica aquí.


In [ ]:
qc_b3 = json.loads((SD / 'stage01c_qc.json').read_text(encoding='utf-8'))
OBJECT_YX = tuple(float(v) for v in qc_b3['companion']['pos_yx'])
STAR_YX   = tuple(float(v) for v in qc_b3['primary']['pos_yx'])

psf_path = SD / 'psf_model.json'
PSF_MODEL = json.loads(psf_path.read_text(encoding='utf-8')) if psf_path.exists() else None

# ¿wings-intact? Misma condición que la etapa.
use_raw = (PSF_MODEL is not None and str(APCORR_MODE).lower() in ('auto', 'psf_growth_curve')
           and WINGS_INTACT and (SD / 'stage02_xcorr_cube_stack.fits').exists())
CUBE_PATH = SD / ('stage02_xcorr_cube_stack.fits' if use_raw else 'cube_residual_local_object.fits')
ANNULUS = list(ANNULUS_BKG_PX) if use_raw else None

with fits.open(CUBE_PATH) as h:
    if 'CUBES' in h:
        CUBE = np.asarray(h['CUBES'].data, dtype=float)
        WAVE = np.asarray(h['WAVELENGTH'].data, dtype=float)
    else:
        CUBE = np.asarray(h[0].data, dtype=float)
        WAVE = np.asarray(fits.getdata(SD / 'stage02_xcorr_cube_stack.fits', 'WAVELENGTH'), dtype=float)
if CUBE.ndim == 4:
    CUBE = CUBE[0]

# STAT y sus factores (A4/M5 + B1): el STAT crudo subestima el ruido de apertura.
qc00 = json.loads((SD / 'stage00q_qc.json').read_text(encoding='utf-8'))
qc01 = json.loads((SD / 'stage01_qc.json').read_text(encoding='utf-8'))
m5 = qc00.get('m5_stat', {})
STAT_FACTOR = float(CFG.get('x01_stat_factor_box3', m5.get('factor_box3_median', 1.0)) or 1.0)
COV_FACTOR  = float(CFG.get('x01_covariance_factor_box3',
                            qc01.get('stat', {}).get('covariance_factor_box3', 1.0)) or 1.0)
STAT_STATUS = str(CFG.get('x01_stat_status', m5.get('status', 'unknown')))
stat_src = Path(CFG.get('x01_stat_cube_fits') or (SD / 'stage02_xcorr_cube_stack.fits'))
STAT_CUBE = None
if stat_src.exists():
    with fits.open(stat_src, memmap=True) as h:
        if 'STAT' in h:
            s = np.asarray(h['STAT'].data, dtype=float)
            STAT_CUBE = s[0] if s.ndim == 4 else s
    if STAT_CUBE is not None and STAT_CUBE.shape != CUBE.shape:
        print('STAT descartado por forma:', STAT_CUBE.shape, '!=', CUBE.shape); STAT_CUBE = None

# La unidad, con la MISMA regla que usa la cadena: el stack de B2 no
# lleva BUNIT (se escribió antes de que B1/B2 lo propagaran), así que
# `resolve_bunit` cae al cubo de entrada del run. Leer solo la
# cabecera del stack dejaba las colorbars sin unidad.
from musepipe.io import resolve_bunit
_stack_bunit = str(fits.getheader(CUBE_PATH, 0).get('BUNIT', '') or
                   fits.getheader(CUBE_PATH, 1).get('BUNIT', '')) or None
BUNIT = resolve_bunit(X01, stack_bunit=_stack_bunit)
UNIDAD = BUNIT or 'sin unidad declarada'   # etiqueta de las colorbars
print('cubo      :', CUBE_PATH.name, CUBE.shape, '| wings-intact:', use_raw,
      '| BUNIT:', BUNIT or 'sin declarar')
print('fondo     :', 'anillo ' + str(ANNULUS) if ANNULUS else 'ninguno (residual 04b)')
print('compañero :', [round(v, 2) for v in OBJECT_YX], ' primaria:', [round(v, 2) for v in STAR_YX])
print('PSF       :', (PSF_MODEL or {}).get('form', 'sin modelo'))
print(f'STAT      : factor={STAT_FACTOR:.3f} covarianza={COV_FACTOR:.3f} estado={STAT_STATUS}')


## 3 · Las funciones numéricas, copiadas de `musepipe`

Copia **literal** del fuente, para que puedas editarla. Todo lo que viene después usa estos nombres locales, así que un cambio aquí se propaga al resultado — y la comparación del final lo cuantifica.

- `finite_values` — de `musepipe/stats.py`
- `robust_sigma` — de `musepipe/stats.py`
- `robust_sigma_axis0` — de `musepipe/stats.py`
- `angular_separation_deg` — de `musepipe/apertures.py`
- `aperture_weights` — de `musepipe/apertures.py`
- `same_radius_control_positions` — de `musepipe/apertures.py`
- `_as_cube` — de `musepipe/extraction/aperture.py`
- `_npix_eff` — de `musepipe/extraction/aperture.py`
- `aperture_spectrum` — de `musepipe/extraction/aperture.py`
- `annulus_background_spectrum` — de `musepipe/extraction/aperture.py`
- `aperture_stat_error` — de `musepipe/extraction/aperture.py`
- `control_aperture_spectra` — de `musepipe/extraction/aperture.py`
- `_flag_window` — de `musepipe/extraction/aperture.py`
- `channel_flags` — de `musepipe/extraction/aperture.py`
- `aperture_correction_from_psf` — de `musepipe/extraction/aperture.py`


In [ ]:
# ------------------------------------------------------------------
# COPIA EDITABLE. Fuente: musepipe (ver el chequeo de deriva abajo).
# ------------------------------------------------------------------
from musepipe.psf import evaluate_psf_model
from typing import Sequence
import math
import numpy as np
import warnings
# `evaluate_psf_model` viene de C1 y `run_channel_chunks` es paralelismo:
# no son lo que se ajusta aquí, por eso se importan en vez de copiarse.

FLAG_BAD_WINDOW = 1
FLAG_SKYLINE = 2
FLAG_INTERPOLATED = 4
FLAG_CLIPPED = 8


def finite_values(values) -> np.ndarray:
    """Return finite values as a float64 1D array."""

    arr = np.asarray(values, dtype=np.float64)
    return arr[np.isfinite(arr)]


def robust_sigma(values) -> float:
    """Robust 1D sigma estimate using MAD with std fallback."""

    vals = finite_values(values)
    if vals.size == 0:
        return np.nan
    med = np.nanmedian(vals)
    mad = np.nanmedian(np.abs(vals - med))
    sigma = 1.4826 * mad
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = np.nanstd(vals)
    return float(sigma)


def robust_sigma_axis0(values) -> np.ndarray:
    """Robust sigma along axis 0 using MAD with std fallback per column."""

    arr = np.asarray(values, dtype=np.float64)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        med = np.nanmedian(arr, axis=0)
        mad = np.nanmedian(np.abs(arr - med[None, :]), axis=0)
        sigma = 1.4826 * mad
        std = np.nanstd(arr, axis=0)
    bad = ~np.isfinite(sigma) | (sigma <= 0)
    sigma[bad] = std[bad]
    return sigma


def angular_separation_deg(a, b) -> float:
    """Smallest angular separation between two angles in radians, in degrees."""

    return abs(math.degrees(math.atan2(math.sin(a - b), math.cos(a - b))))


def aperture_weights(ny, nx, center_yx, aperture) -> np.ndarray:
    """Build a 2D aperture-weight image."""

    y0, x0 = map(float, center_yx)
    yy, xx = np.mgrid[:ny, :nx]
    rr2 = (yy - y0) ** 2 + (xx - x0) ** 2
    weights = np.zeros((ny, nx), dtype=np.float64)
    kind = aperture["kind"]

    if kind == "pixel":
        y = int(round(y0))
        x = int(round(x0))
        if 0 <= y < ny and 0 <= x < nx:
            weights[y, x] = 1.0
    elif kind == "box":
        size = int(aperture.get("size", 3))
        half = size // 2
        y = int(round(y0))
        x = int(round(x0))
        y1 = max(0, y - half)
        y2 = min(ny, y + half + 1)
        x1 = max(0, x - half)
        x2 = min(nx, x + half + 1)
        weights[y1:y2, x1:x2] = 1.0
    elif kind == "circle":
        radius = float(aperture["radius_px"])
        weights[rr2 <= radius**2] = 1.0
    elif kind == "gaussian":
        sigma = float(aperture["sigma_px"])
        radius = float(aperture.get("radius_px", 3.0 * sigma))
        mask = rr2 <= radius**2
        weights[mask] = np.exp(-0.5 * rr2[mask] / sigma**2)
    else:
        raise ValueError(f"Unknown aperture kind: {kind}")
    return weights


def same_radius_control_positions(
    object_yx,
    star_yx,
    ny,
    nx,
    n_positions=8,
    exclude_angle_deg=25.0,
    margin_px=4,
):
    """Return integer control positions at the same star-object radius."""

    oy, ox = map(float, object_yx)
    sy, sx = map(float, star_yx)
    dy = oy - sy
    dx = ox - sx
    radius = math.hypot(dy, dx)
    theta0 = math.atan2(dy, dx)

    controls = []
    for k in range(int(n_positions)):
        theta = theta0 + 2.0 * math.pi * k / float(n_positions)
        if angular_separation_deg(theta, theta0) < exclude_angle_deg:
            continue
        y = int(round(sy + radius * math.sin(theta)))
        x = int(round(sx + radius * math.cos(theta)))
        if margin_px <= y < ny - margin_px and margin_px <= x < nx - margin_px:
            controls.append((y, x))
    return controls


def _as_cube(cube_zyx, name="cube") -> np.ndarray:
    cube = np.asarray(cube_zyx, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected {name} with shape (nz,ny,nx), got {cube.shape}.")
    return cube


def _npix_eff(cube_zyx: np.ndarray, weights: np.ndarray) -> np.ndarray:
    valid = np.isfinite(cube_zyx) & (weights[None, :, :] > 0)
    sumw = np.sum(weights[None, :, :] * valid, axis=(1, 2))
    sumw2 = np.sum((weights[None, :, :] ** 2) * valid, axis=(1, 2))
    out = np.full(cube_zyx.shape[0], np.nan, dtype=np.float64)
    good = sumw2 > 0
    out[good] = (sumw[good] ** 2) / sumw2[good]
    return out


def aperture_spectrum(cube_zyx, center_yx, aperture: dict) -> tuple[np.ndarray, np.ndarray]:
    """Return weighted-sum spectrum and per-channel effective pixel count."""

    cube = _as_cube(cube_zyx)
    _, ny, nx = cube.shape
    weights = aperture_weights(ny, nx, center_yx, aperture)
    weighted = cube * weights[None, :, :]
    with np.errstate(invalid="ignore"):
        flux = np.nansum(weighted, axis=(1, 2)).astype(np.float64)
    npix_eff = _npix_eff(cube, weights)
    flux[~np.isfinite(npix_eff)] = np.nan
    return flux, npix_eff


def annulus_background_spectrum(cube_zyx, center_yx, r_in, r_out, *, exclude_yx=None, exclude_radius=0.0):
    """Per-channel local background = median of a source-free annulus.

    Used for the wings-intact aperture-correction path: subtracting a distant
    annulus (rather than a local surface, stage04b) preserves the companion's
    PSF wings so the PSF growth-curve aperture correction stays self-consistent
    (box3<box5). Excludes a region around ``exclude_yx`` (the primary)."""

    cube = _as_cube(cube_zyx)
    _, ny, nx = cube.shape
    yy, xx = np.mgrid[0:ny, 0:nx]
    r = np.hypot(yy - float(center_yx[0]), xx - float(center_yx[1]))
    mask = (r >= float(r_in)) & (r <= float(r_out))
    if exclude_yx is not None and float(exclude_radius) > 0:
        mask &= np.hypot(yy - float(exclude_yx[0]), xx - float(exclude_yx[1])) > float(exclude_radius)
    if not mask.any():
        return np.zeros(cube.shape[0], dtype=np.float64)
    vals = cube[:, mask]
    with np.errstate(all="ignore"):
        return np.nanmedian(vals, axis=1).astype(np.float64)


def aperture_stat_error(
    stat_zyx,
    center_yx,
    aperture: dict,
    *,
    stat_factor: float = 1.0,
    covariance_factor: float = 1.0,
) -> np.ndarray:
    """Propagate a variance cube through the same aperture weights."""

    stat = _as_cube(stat_zyx, name="stat")
    _, ny, nx = stat.shape
    weights = aperture_weights(ny, nx, center_yx, aperture)
    valid = np.isfinite(stat) & (weights[None, :, :] > 0)
    variance = np.nansum(stat * (weights[None, :, :] ** 2), axis=(1, 2))
    variance[np.sum(valid, axis=(1, 2)) == 0] = np.nan
    factor = float(stat_factor) * float(covariance_factor)
    if not np.isfinite(factor) or factor <= 0:
        factor = 1.0
    variance *= factor
    return np.sqrt(np.clip(variance, 0.0, np.inf)).astype(np.float64)


def control_aperture_spectra(
    cube_zyx,
    object_yx,
    star_yx,
    aperture: dict,
    *,
    n_controls: int = 8,
    exclude_angle_deg: float = 25.0,
    margin_px: int | None = None,
) -> tuple[list[tuple[int, int]], np.ndarray]:
    cube = _as_cube(cube_zyx)
    _, ny, nx = cube.shape
    if margin_px is None:
        if str(aperture.get("kind", "box")) == "box":
            margin_px = int(aperture.get("size", 3)) // 2 + 1
        else:
            margin_px = int(math.ceil(float(aperture.get("radius_px", 3.0)))) + 1
    controls = same_radius_control_positions(
        object_yx,
        star_yx,
        ny,
        nx,
        n_positions=int(n_controls),
        exclude_angle_deg=float(exclude_angle_deg),
        margin_px=int(margin_px),
    )
    spectra = []
    npix = []
    for yx in controls:
        flux, npix_eff = aperture_spectrum(cube, yx, aperture)
        spectra.append(flux)
        npix.append(npix_eff)
    if not spectra:
        empty = np.empty((0, cube.shape[0]), dtype=np.float64)
        return controls, empty, empty.copy()
    return controls, np.asarray(spectra, dtype=np.float64), np.asarray(npix, dtype=np.float64)


def _flag_window(wave_A: np.ndarray, windows_A: Sequence[Sequence[float]], bit: int, flags: np.ndarray) -> None:
    for window in windows_A or ():
        if window is None or len(window) != 2:
            continue
        lo, hi = window
        if lo is None or hi is None:
            continue
        flags[(wave_A >= float(lo)) & (wave_A <= float(hi))] |= int(bit)


def channel_flags(
    wave_A,
    *,
    bad_windows_A: Sequence[Sequence[float]] = (),
    skyline_windows_A: Sequence[Sequence[float]] = (),
    interpolated_windows_A: Sequence[Sequence[float]] = (),
    clipped_mask=None,
    good_mask=None,
    bad_mask=None,
) -> np.ndarray:
    wave = np.asarray(wave_A, dtype=np.float64)
    flags = np.zeros(wave.size, dtype=np.int32)
    _flag_window(wave, bad_windows_A, FLAG_BAD_WINDOW, flags)
    _flag_window(wave, skyline_windows_A, FLAG_SKYLINE, flags)
    _flag_window(wave, interpolated_windows_A, FLAG_INTERPOLATED, flags)
    if good_mask is not None:
        flags[~np.asarray(good_mask, dtype=bool)] |= FLAG_BAD_WINDOW
    if bad_mask is not None:
        flags[np.asarray(bad_mask, dtype=bool)] |= FLAG_BAD_WINDOW
    if clipped_mask is not None:
        flags[np.asarray(clipped_mask, dtype=bool)] |= FLAG_CLIPPED
    return flags


def aperture_correction_from_psf(
    wave_A,
    aperture: dict,
    psf_model: dict | None,
    *,
    center_yx=(0.0, 0.0),
    correction_mode: str = "auto",
) -> tuple[np.ndarray, str, float]:
    """Return wavelength-dependent aperture correction from a C1 PSF model."""

    wave = np.asarray(wave_A, dtype=np.float64)
    mode = str(correction_mode or "auto").lower()
    if mode in {"none", "off", "false"}:
        return np.ones(wave.size, dtype=np.float64), "none", 0.0
    if psf_model is None:
        if mode in {"auto", "optional"}:
            return np.ones(wave.size, dtype=np.float64), "none", 0.0
        raise RuntimeError("Aperture correction requested but no psf_model was supplied.")

    norm_radius = float(psf_model.get("norm_radius_px", 25.0))
    half = int(math.ceil(norm_radius))
    frac_y = float(center_yx[0]) - round(float(center_yx[0]))
    frac_x = float(center_yx[1]) - round(float(center_yx[1]))
    source_center = (half + frac_y, half + frac_x)
    yy, xx = np.indices((2 * half + 1, 2 * half + 1), dtype=np.float64)
    dy = yy - source_center[0]
    dx = xx - source_center[1]
    weights = aperture_weights(2 * half + 1, 2 * half + 1, source_center, aperture)
    fractions = np.empty(wave.size, dtype=np.float64)
    for i, w in enumerate(wave):
        psf = evaluate_psf_model(psf_model, float(w), dy, dx)
        frac = float(np.nansum(psf * weights))
        if not np.isfinite(frac) or frac <= 0:
            raise RuntimeError(f"Invalid aperture PSF fraction at wave={w}.")
        fractions[i] = frac
    return (1.0 / fractions).astype(np.float64), "psf_growth_curve", norm_radius


## 4 · Chequeo de deriva

Compara el fuente copiado arriba con el que **hoy** tiene `musepipe`. Si alguien cambió la cadena, esta celda lo dice nombrando la función: es lo que evita que este notebook siga dando resultados «de la cadena» cuando ya no lo son.


In [ ]:
import ast as _ast, hashlib as _hashlib

_SHAS = {
    "musepipe/stats.py:finite_values": "8aa861655f2b",
    "musepipe/stats.py:robust_sigma": "ef2aa72a72de",
    "musepipe/stats.py:robust_sigma_axis0": "0b976272de28",
    "musepipe/apertures.py:angular_separation_deg": "ce3b83206746",
    "musepipe/apertures.py:aperture_weights": "d8e1fd8bb88d",
    "musepipe/apertures.py:same_radius_control_positions": "fb89fcf9dd7d",
    "musepipe/extraction/aperture.py:_as_cube": "67ede036d305",
    "musepipe/extraction/aperture.py:_npix_eff": "32ecf15dcce9",
    "musepipe/extraction/aperture.py:aperture_spectrum": "214c68e30b47",
    "musepipe/extraction/aperture.py:annulus_background_spectrum": "68ec4c4e7299",
    "musepipe/extraction/aperture.py:aperture_stat_error": "a72d7d66c6e6",
    "musepipe/extraction/aperture.py:control_aperture_spectra": "d8a9a0f28448",
    "musepipe/extraction/aperture.py:_flag_window": "7ba686789970",
    "musepipe/extraction/aperture.py:channel_flags": "dbceb28a0a0b",
    "musepipe/extraction/aperture.py:aperture_correction_from_psf": "8ed04307abec"
}

def chequeo_de_deriva(shas=_SHAS, root=ROOT):
    problemas = []
    for key, sha in shas.items():
        rel, name = key.rsplit(':', 1)
        text = (root / rel).read_text(encoding='utf-8')
        lines = text.splitlines(keepends=True)
        node = next((n for n in _ast.parse(text).body
                     if isinstance(n, (_ast.FunctionDef, _ast.ClassDef)) and n.name == name),
                    None)
        if node is None:
            problemas.append(f'{key}: ya no existe en musepipe'); continue
        start = min([node.lineno] + [d.lineno for d in node.decorator_list]) - 1
        src = ''.join(lines[start:node.end_lineno]).rstrip('\n')
        actual = _hashlib.sha256(src.encode('utf-8')).hexdigest()[:12]
        if actual != sha:
            problemas.append(f'{key}: la copia es {sha}, musepipe tiene {actual}')
    return problemas

_deriva = chequeo_de_deriva()
if _deriva:
    print('DERIVA — la cadena cambió y esta copia se quedó atrás:')
    for p in _deriva:
        print('  ·', p)
    print(f'\nRegenera: python scripts/build_debug_notebooks.py --target {TARGET} C2')
else:
    print(f'sin deriva: las {len(_SHAS)} piezas copiadas son las de musepipe')


## 5 · Paso 1 — dónde se mide

Antes de sumar nada, ver el sitio. Dos vistas del **mismo** dato, con escalas distintas a propósito:

- **izquierda**, el campo entero en escala **logarítmica**: así se ve el halo de la primaria, que es lo que domina y lo que hay que quitar. El compañero es invisible aquí, y esa es la lección.
- **derecha**, un zoom sobre el compañero **con el halo ya restado** (el residual de 04b) y escala por percentiles: ahora sí se ve la fuente que estamos midiendo.

En las dos, la caja de extracción; en la izquierda, además, el anillo de fondo y la primaria. La imagen es la **mediana en λ** (1 de cada 20 canales), y el zoom la toma solo en el rojo (7500–9000 Å), que es donde el compañero se detecta.


In [ ]:
from matplotlib.colors import LogNorm
from matplotlib.patches import Circle, Rectangle

weights = aperture_weights(CUBE.shape[1], CUBE.shape[2], OBJECT_YX, APERTURE)
raw_flux, npix_eff = aperture_spectrum(CUBE, OBJECT_YX, APERTURE)
print('píxeles con peso:', int((weights > 0).sum()),
      '| npix_eff mediano:', float(np.nanmedian(npix_eff)))

campo = np.nanmedian(CUBE[::20], axis=0)
rojo_ch = (WAVE >= 7500) & (WAVE <= 9000)
resid_path = SD / 'cube_residual_local_object.fits'
zoom_src = (np.nanmedian(np.asarray(fits.getdata(resid_path), float)[rojo_ch][::10], axis=0)
            if resid_path.exists() else campo)
titulo_zoom = 'halo restado (residual 04b)' if resid_path.exists() else 'sin residual 04b en disco'

yc, xc = OBJECT_YX
size = float(APERTURE['size'])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
pos = campo[np.isfinite(campo) & (campo > 0)]
a1.imshow(campo, origin='lower', cmap='magma',
          norm=LogNorm(vmin=np.percentile(pos, 60), vmax=np.percentile(pos, 99.9)))
a1.add_patch(Rectangle((xc - size/2 - 0.5, yc - size/2 - 0.5), size, size,
                       fill=False, edgecolor='tab:cyan', lw=1.6))
if ANNULUS is not None:
    for r, ls_ in ((ANNULUS[0], '-'), (ANNULUS[1], '-')):
        a1.add_patch(Circle((xc, yc), r, fill=False, edgecolor='tab:cyan', lw=0.9, ls=ls_, alpha=0.8))
a1.plot(STAR_YX[1], STAR_YX[0], marker='+', ms=11, mew=1.6, color='w')
a1.annotate('primaria', (STAR_YX[1], STAR_YX[0]), textcoords='offset points',
            xytext=(0, 9), ha='center', fontsize=7, color='w')
a1.annotate('compañero', (xc, yc), textcoords='offset points', xytext=(0, 10),
            ha='center', fontsize=7, color='tab:cyan')
im1 = a1.get_images()[0]
cb1 = fig.colorbar(im1, ax=a1, shrink=0.82)
cb1.set_label(f'flujo mediano [{UNIDAD}] · escala LOG', fontsize=7)
a1.set_title(f'campo, escala log ({np.percentile(pos, 60):.3g}–{np.percentile(pos, 99.9):.3g}):'
             ' manda el halo (el compañero no se ve)', fontsize=9)
a1.set_xlabel('x [px]'); a1.set_ylabel('y [px]')

h = 12
y0, x0 = int(round(yc)) - h, int(round(xc)) - h
recorte = zoom_src[y0:y0 + 2*h + 1, x0:x0 + 2*h + 1]
fin_r = recorte[np.isfinite(recorte)]
a2.imshow(recorte, origin='lower', cmap='viridis',
          vmin=np.percentile(fin_r, 5), vmax=np.percentile(fin_r, 99.5),
          extent=[x0 - 0.5, x0 + 2*h + 0.5, y0 - 0.5, y0 + 2*h + 0.5])
a2.add_patch(Rectangle((xc - size/2 - 0.5, yc - size/2 - 0.5), size, size,
                       fill=False, edgecolor='tab:red', lw=1.8))
im2 = a2.get_images()[0]
cb2 = fig.colorbar(im2, ax=a2, shrink=0.82)
cb2.set_label(f'flujo mediano [{UNIDAD}] · escala LINEAL', fontsize=7)
a2.set_title(f'zoom al compañero · {titulo_zoom}\nescala lineal, percentiles 5–99.5 ({np.percentile(fin_r, 5):.3g}–{np.percentile(fin_r, 99.5):.3g}) · mediana en 7500–9000 Å',
             fontsize=9)
a2.set_xlabel('x [px]')
fig.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(11, 2.6))
ax.plot(WAVE, npix_eff, lw=0.8)
ax.set_xlabel('λ [Å]'); ax.set_ylabel('npix_eff')
ax.set_title('píxeles efectivos por canal: no son 9 fijos, bajan donde hay NaN', fontsize=9)
fig.tight_layout(); plt.show()


## 6 · Paso 2 — el fondo de anillo, y dónde se mide

Solo cuando la extracción es *wings-intact* (cubo crudo). Se toma la **mediana por canal** de los píxeles del anillo entre `ANNULUS[0]` y `ANNULUS[1]` alrededor del compañero, **excluyendo** un disco de `ANNULUS[2]` px alrededor de la primaria — sin esa exclusión el anillo mediría el halo de la estrella, no el fondo local.

Se resta multiplicado por `npix_eff`, para pasar de fondo **por píxel** a fondo **de la apertura**. La figura enseña exactamente qué píxeles entran.


In [ ]:
if ANNULUS is not None:
    bkg = annulus_background_spectrum(CUBE, OBJECT_YX, ANNULUS[0], ANNULUS[1],
                                      exclude_yx=STAR_YX,
                                      exclude_radius=(ANNULUS[2] if len(ANNULUS) > 2 else 30.0))
    raw_flux_bkgsub = raw_flux - bkg * npix_eff
    print(f'fondo mediano por píxel      : {float(np.nanmedian(bkg)):8.3f}')
    print(f'resta mediana a la apertura  : {float(np.nanmedian(bkg * npix_eff)):8.2f}'
          f'  (= fondo × npix_eff)')
    # Los mismos pixeles que usa `annulus_background_spectrum`, dibujados.
    yy, xx = np.indices(CUBE.shape[1:], dtype=float)
    rr = np.hypot(yy - OBJECT_YX[0], xx - OBJECT_YX[1])
    rr_star = np.hypot(yy - STAR_YX[0], xx - STAR_YX[1])
    anillo = (rr >= ANNULUS[0]) & (rr <= ANNULUS[1])
    excl = rr_star <= (ANNULUS[2] if len(ANNULUS) > 2 else 30.0)
    usados = anillo & ~excl
    print(f'píxeles del anillo           : {int(anillo.sum())}'
          f' | usados tras excluir la primaria: {int(usados.sum())}')
    h2 = int(ANNULUS[1]) + 4
    y0, x0 = int(round(OBJECT_YX[0])) - h2, int(round(OBJECT_YX[1])) - h2
    sl = (slice(max(y0, 0), y0 + 2*h2 + 1), slice(max(x0, 0), x0 + 2*h2 + 1))
    fig, (b1, b2) = plt.subplots(1, 2, figsize=(11, 4.2))
    img = np.nanmedian(CUBE[::20], axis=0)[sl]
    fin_i = img[np.isfinite(img)]
    b1.imshow(img, origin='lower', cmap='magma',
              vmin=np.percentile(fin_i, 5), vmax=np.percentile(fin_i, 99))
    cbb = fig.colorbar(b1.get_images()[0], ax=b1, shrink=0.8)
    cbb.set_label(f'flujo mediano [{UNIDAD}] · lineal', fontsize=7)
    b1.set_title(f'dato (mediana en λ) · escala lineal, percentiles 5–99'
                 f' ({np.percentile(fin_i, 5):.3g}–{np.percentile(fin_i, 99):.3g})', fontsize=9)
    mascara = np.where(usados[sl], 1.0, np.where(anillo[sl], 0.4, np.nan))
    b2.imshow(img, origin='lower', cmap='gray',
              vmin=np.percentile(fin_i, 5), vmax=np.percentile(fin_i, 99))
    b2.imshow(mascara, origin='lower', cmap='cool', alpha=0.55, vmin=0, vmax=1)
    b2.set_title('anillo: en claro lo usado, en oscuro lo excluido\n(disco de la primaria)',
                 fontsize=9)
    fig.tight_layout(); plt.show()
else:
    bkg = None
    raw_flux_bkgsub = raw_flux
    print('sin fondo de anillo (se extrae del residual de 04b)')
raw_flux = raw_flux_bkgsub


## 7 · Paso 3 — controles y error empírico

**La regla que gobierna todo el modelo de ruido**: σ no sale del STAT del cubo, sale de medir **lo mismo donde no hay nada**. Los controles son aperturas idénticas (misma caja, mismo radio a la primaria, mismo fondo de anillo) repartidas en ángulo, excluyendo un cono alrededor del compañero.

**Cómo se calcula σ empírico**, canal a canal:

1. se extrae el espectro de cada uno de los N controles, con el mismo procedimiento;
2. para **cada canal**, se mira la dispersión de esos N valores;
3. esa dispersión se mide con `robust_sigma_axis0`, que usa la **MAD** (mediana de las desviaciones absolutas × 1.4826) en vez de la desviación típica, para que un control contaminado no infle σ.

Es decir: σ(λ) **no** es la dispersión del objeto a lo largo de λ, sino la dispersión **entre posiciones** en ese canal. Ver [`docs/noise_model.md`](../../../docs/noise_model.md).

En la figura: **cada línea gris es un control** (su espectro completo); la azul es el **objeto**, que es la **suma de la apertura box3** —no el píxel más brillante—, ya con el fondo restado; la roja es el σ que sale del paso 3.


In [ ]:
controls_yx, control_spectra, control_npix = control_aperture_spectra(
    CUBE, OBJECT_YX, STAR_YX, APERTURE,
    n_controls=N_CONTROLS, exclude_angle_deg=EXCLUDE_ANGLE_DEG)
control_bkgsub = control_spectra
if ANNULUS is not None and control_spectra.shape[0]:
    control_bkgsub = control_spectra.copy()
    for k, yx in enumerate(controls_yx):
        cb = annulus_background_spectrum(CUBE, yx, ANNULUS[0], ANNULUS[1],
                                        exclude_yx=STAR_YX,
                                        exclude_radius=(ANNULUS[2] if len(ANNULUS) > 2 else 30.0))
        control_bkgsub[k] = control_spectra[k] - cb * control_npix[k]
if control_bkgsub.shape[0] >= 2:
    raw_err_emp = robust_sigma_axis0(control_bkgsub)
else:
    raw_err_emp = np.full(WAVE.size, robust_sigma(raw_flux), dtype=float)
print(f'{len(controls_yx)} controles a {np.hypot(*(np.array(OBJECT_YX) - np.array(STAR_YX))):.1f} px'
      f' de la primaria | σ empírico mediano = {float(np.nanmedian(raw_err_emp)):.2f}')
print('σ(λ) = dispersión ENTRE los controles en ese canal (MAD robusta), no a lo largo de λ')

# Donde estan los N controles: mismo radio a la primaria, repartidos en
# angulo, con un cono excluido alrededor del compañero.
campo_c = np.nanmedian(CUBE[::20], axis=0)
pos_c = campo_c[np.isfinite(campo_c) & (campo_c > 0)]
sep_px = float(np.hypot(*(np.array(OBJECT_YX) - np.array(STAR_YX))))
figc, axc = plt.subplots(figsize=(6.6, 6.0))
imc = axc.imshow(campo_c, origin='lower', cmap='magma',
                 norm=LogNorm(vmin=np.percentile(pos_c, 60), vmax=np.percentile(pos_c, 99.9)))
cbc = figc.colorbar(imc, ax=axc, shrink=0.82)
cbc.set_label('flujo mediano · escala LOG', fontsize=7)
axc.add_patch(Circle((STAR_YX[1], STAR_YX[0]), sep_px, fill=False,
                     edgecolor='w', lw=0.8, ls=':', alpha=0.6))
axc.plot(STAR_YX[1], STAR_YX[0], marker='+', ms=11, mew=1.6, color='w')
axc.add_patch(Rectangle((OBJECT_YX[1] - size/2 - 0.5, OBJECT_YX[0] - size/2 - 0.5),
                        size, size, fill=False, edgecolor='tab:cyan', lw=1.8))
axc.annotate('compañero', (OBJECT_YX[1], OBJECT_YX[0]), textcoords='offset points',
             xytext=(0, 10), ha='center', fontsize=7, color='tab:cyan')
for k, (cy, cx) in enumerate(controls_yx):
    axc.add_patch(Rectangle((cx - size/2 - 0.5, cy - size/2 - 0.5), size, size,
                            fill=False, edgecolor='tab:green', lw=1.2))
    axc.annotate(str(k), (cx, cy), textcoords='offset points', xytext=(0, 7),
                 ha='center', fontsize=6, color='tab:green')
axc.set_title(f'los {len(controls_yx)} controles: mismo radio ({sep_px:.0f} px) que el compañero,'
              f'\ncono de ±{EXCLUDE_ANGLE_DEG:.0f}° excluido a su alrededor', fontsize=9)
axc.set_xlabel('x [px]'); axc.set_ylabel('y [px]')
figc.tight_layout(); plt.show()

from musepipe.reduction.telluric import TELLURIC_BANDS
fig, ax = plt.subplots(figsize=(11, 3.6))
for j, (lo, hi) in enumerate(BAD_WINDOWS_A):
    ax.axvspan(lo, hi, color='0.85', zorder=0,
               label='ventana ignorada (flags)' if j == 0 else None)
for j, (nombre, (lo, hi)) in enumerate(TELLURIC_BANDS.items()):
    ax.axvspan(lo, hi, color='tab:orange', alpha=0.13, zorder=0,
               label='bandas telúricas (A3)' if j == 0 else None)
    ax.annotate(nombre, ((lo + hi) / 2, 0.97), xycoords=('data', 'axes fraction'),
                ha='center', va='top', fontsize=6, color='tab:orange')
for i, c in enumerate(control_bkgsub):
    ax.plot(WAVE, c, lw=0.4, alpha=0.35, color='0.6',
            label=f'{len(control_bkgsub)} controles (uno por línea)' if i == 0 else None)
ax.plot(WAVE, raw_flux, lw=0.6, color='tab:blue', label='objeto = suma de la caja box3')
ax.plot(WAVE, raw_err_emp, lw=1.0, color='tab:red', label='σ empírico = dispersión entre controles')
ax.set_xlabel('λ [Å]'); ax.legend(fontsize=8)
ax.set_title('el objeto y los controles, procesados igual', fontsize=9)
fig.tight_layout(); plt.show()


## 8 · Paso 4 — error por STAT y corrección de apertura

El STAT del cubo **nunca** se usa crudo: lleva el factor de M5 (el DRS subestima la varianza) y el de covarianza de B1 (el desplazamiento subpíxel correlacionó píxeles vecinos, así que sumar N píxeles no da √N).

### Qué es `apcorr` (y qué NO es)

**No es un porcentaje: es un factor multiplicativo adimensional.** Si `f` es la fracción de la PSF que cae dentro de la apertura, entonces

> `apcorr = 1 / f`,  y por tanto  `f = 1 / apcorr`

Una caja 3×3 en NFM recoge `f ≈ 0.022`, o sea el **2.2%** de la luz de la fuente, así que `apcorr ≈ 45`. Leído al revés: **`apcorr = 100` significaría que la caja recoge el 1%**, no el 100%. Cuanto mayor el número, *menos* luz entra en la apertura.

Y no, no es «el flujo que llega de la fuente a ese lugar»: es el factor con el que hay que multiplicar lo medido para recuperar el flujo **total** de la fuente.

- **De dónde sale**: del modelo de PSF de **C1** (`psf_model.json`). Para cada λ se evalúa la PSF, se suma dentro de la **misma máscara de apertura** y se toma `apcorr = 1 / fracción`. La PSF está normalizada dentro de `norm_radius_px` (25 px), así que «flujo total» significa *el que hay dentro de ese radio*.
- **Depende de λ** porque la PSF se ensancha hacia el azul: más luz fuera de la caja, corrección mayor.
- **Cómo se aplica**: multiplicando, a **flujo y errores por igual** (`flux = raw × apcorr`), así que no cambia la relación señal-ruido; solo la escala.
- Si no hay modelo de PSF, `apcorr = 1` y el producto lo declara (`APCMODE=none`) en vez de fingir una corrección.


In [ ]:
stat_usable = (STAT_CUBE is not None and str(ERROR_MODE).lower() != 'empirical'
               and STAT_STATUS.lower() != 'red')
if stat_usable:
    raw_err = aperture_stat_error(STAT_CUBE, OBJECT_YX, APERTURE,
                                  stat_factor=STAT_FACTOR, covariance_factor=COV_FACTOR)
    error_mode = 'stat'
else:
    raw_err = np.asarray(raw_err_emp, dtype=float)
    error_mode = 'empirical'

apcorr, apcorr_mode, norm_radius = aperture_correction_from_psf(
    WAVE, APERTURE, PSF_MODEL, center_yx=OBJECT_YX, correction_mode=APCORR_MODE)
flags = channel_flags(WAVE, bad_windows_A=BAD_WINDOWS_A, skyline_windows_A=SKYLINE_WINDOWS_A,
                      interpolated_windows_A=INTERPOLATED_WIN_A)
print(f'error: modo={error_mode} (STAT {STAT_STATUS})')
print(f'apcorr: modo={apcorr_mode} | mediana={float(np.nanmedian(apcorr)):.1f}'
      f' -> la caja recoge el {100 / float(np.nanmedian(apcorr)):.2f}% de la PSF'
      f' | norm_radius={norm_radius:g} px')
print(f'canales marcados por flags: {int((flags != 0).sum())}')

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(WAVE, apcorr, lw=1.0, color='tab:blue')
ax.set_xlabel('λ [Å]'); ax.set_ylabel('apcorr  [adimensional]', color='tab:blue')
ax.tick_params(axis='y', labelcolor='tab:blue')
# El mismo dato leído como fracción de PSF capturada, que es lo intuitivo.
ax2 = ax.twinx()
ax2.plot(WAVE, 100.0 / apcorr, lw=0.0)
ax2.set_ylim(100.0 / np.array(ax.get_ylim())[::-1])
ax2.set_ylabel('luz capturada por la caja  [%]', color='tab:grey')
ax2.tick_params(axis='y', labelcolor='tab:grey')
ax.set_title('corrección de apertura vs λ: sube al azul porque la PSF se ensancha'
             ' (más luz fuera de la caja)', fontsize=9)
fig.tight_layout(); plt.show()


## 9 · Paso 5 — el espectro, binado con su error

El flujo por canal es demasiado ruidoso para leerse, y una mediana móvil suaviza pero **no dice cuánto vale lo que se ve**. Aquí se **bina**: se agrupan `BIN_CANALES` canales, el flujo es la media y el error se propaga como gaussiano independiente,

> σ_bin = √(Σ σᵢ²) / n

que para σ constante es el conocido σ/√n. Así cada punto lleva su barra y se puede juzgar si el espectro está por encima de cero.

> **Aviso que la propia cadena mide**: esa fórmula supone canales **independientes**, y no lo son. G1 midió `n_eff/n ≈ 0.43` (el remuestreo en λ correlacionó canales vecinos), así que el error binado gaussiano está **subestimado en ~√(1/0.43) ≈ 1.5×**. La celda dibuja las dos barras: la gaussiana y la corregida por `n_eff`.


In [ ]:
BIN_CANALES = 25        # cámbialo y vuelve a ejecutar
N_EFF_OVER_N = 0.43     # medido por G1 (docs/noise_model.md)

flux         = raw_flux * apcorr
flux_err     = raw_err * apcorr
flux_err_emp = raw_err_emp * apcorr

def binea(wave, flux, err, n):
    """Media por bloques de n canales, con error gaussiano independiente."""
    n = int(n)
    corte = (wave.size // n) * n
    w = wave[:corte].reshape(-1, n)
    f = flux[:corte].reshape(-1, n)
    e = err[:corte].reshape(-1, n)
    bueno = np.isfinite(f) & np.isfinite(e)
    cuenta = bueno.sum(axis=1)
    with np.errstate(invalid='ignore', divide='ignore'):
        wb = np.nanmean(np.where(bueno, w, np.nan), axis=1)
        fb = np.nansum(np.where(bueno, f, 0.0), axis=1) / np.maximum(cuenta, 1)
        eb = np.sqrt(np.nansum(np.where(bueno, e, 0.0) ** 2, axis=1)) / np.maximum(cuenta, 1)
    vacio = cuenta == 0
    fb[vacio] = np.nan; eb[vacio] = np.nan
    return wb, fb, eb, cuenta

wb, fb, eb, cuenta = binea(WAVE, flux, flux_err_emp, BIN_CANALES)
eb_corr = eb / np.sqrt(N_EFF_OVER_N)   # canales correlacionados (G1)
rojo_b = (wb >= 7500) & (wb <= 9000)
snr = np.abs(fb) / eb_corr
print(f'binado de {BIN_CANALES} canales -> {np.isfinite(fb).sum()} puntos')
print(f'  flujo mediano en 7500–9000 Å : {np.nanmedian(fb[rojo_b]):9.2f}')
print(f'  error binado gaussiano       : {np.nanmedian(eb[rojo_b]):9.2f}')
print(f'  corregido por n_eff/n={N_EFF_OVER_N:.2f}   : {np.nanmedian(eb_corr[rojo_b]):9.2f}'
      f'  (×{1/np.sqrt(N_EFF_OVER_N):.2f})')
print(f'  S/N mediano en esa banda     : {np.nanmedian(snr[rojo_b]):9.2f}')

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(WAVE, flux, lw=0.3, color='0.75', label='flujo por canal (crudo)')
ax.errorbar(wb, fb, yerr=eb_corr, fmt='o', ms=3, lw=0.9, color='tab:blue',
            ecolor='tab:blue', alpha=0.9,
            label=f'binado {BIN_CANALES} ch, ±σ corregido por n_eff')
ax.errorbar(wb, fb, yerr=eb, fmt='none', lw=1.8, ecolor='tab:orange', alpha=0.8,
            label='±σ gaussiano (subestima: canales correlacionados)')
ax.axhline(0, color='0.5', lw=0.7)
ax.axvline(6563, color='tab:red', ls=':', label='Hα')
ax.set_ylim(*np.nanpercentile(fb[np.isfinite(fb)], [1, 99]) * np.array([2.5, 2.5]))
ax.set_xlabel('λ [Å]'); ax.set_ylabel('flujo (apcorr aplicada)'); ax.legend(fontsize=8)
ax.set_title('C2 rehecho en el notebook, binado con su error', fontsize=9)
fig.tight_layout(); plt.show()


## 10 · Tres controles con geometría fija

Los N controles del paso 3 los coloca el pipeline repartidos en ángulo. Aquí se miran **tres sitios elegidos a mano**, a la misma distancia de la primaria que el compañero: el **opuesto** (PA + 180°) y los dos **perpendiculares** (PA ± 90°). Se colocan con los helpers de B3, así que heredan su convención de PA y el `north_angle_deg` del run.

Cada uno pasa por **exactamente el mismo proceso** que el compañero: misma caja, mismo fondo de anillo, misma `apcorr`. Ahí no hay ninguna fuente, así que lo que se vea es halo residual y ruido — y su dispersión dice cuánto cambia el fondo con el ángulo a esa misma separación.


In [ ]:
from musepipe.stages.stage01c_localize import position_from_sep_pa

astro = qc_b3.get('astrometry') or {}
escala = qc_b3.get('pixel_scale_arcsec')
norte = (qc_b3.get('wcs_orientation') or {}).get('north_angle_deg', 0.0)
TRES = [('opuesto', 180.0), ('perpendicular +90', 90.0), ('perpendicular -90', -90.0)]
fig, axes = plt.subplots(len(TRES), 1, figsize=(11, 2.5 * len(TRES)), sharex=True)
for ax_i, (nombre, delta) in zip(np.atleast_1d(axes), TRES):
    pos = position_from_sep_pa((float(STAR_YX[0]), float(STAR_YX[1])),
                               float(astro['sep_arcsec']),
                               float(astro['pa_deg']) + delta,
                               float(escala), north_angle_deg=float(norte or 0.0))
    f_c, npix_c = aperture_spectrum(CUBE, pos, APERTURE)
    if ANNULUS is not None:
        b_c = annulus_background_spectrum(CUBE, pos, ANNULUS[0], ANNULUS[1],
                                         exclude_yx=STAR_YX,
                                         exclude_radius=(ANNULUS[2] if len(ANNULUS) > 2 else 30.0))
        f_c = f_c - b_c * npix_c
    f_c = f_c * apcorr
    wbc, fbc, ebc, _n = binea(WAVE, f_c, flux_err_emp, BIN_CANALES)
    rb = (wbc >= 7500) & (wbc <= 9000)
    for lo, hi in BAD_WINDOWS_A:
        ax_i.axvspan(lo, hi, color='0.85', zorder=0)
    ax_i.errorbar(wbc, fbc, yerr=ebc / np.sqrt(N_EFF_OVER_N), fmt='o', ms=2.5, lw=0.8,
                  color='tab:green')
    ax_i.axhline(0, color='0.5', lw=0.7)
    pa = (float(astro['pa_deg']) + delta) % 360.0
    ax_i.set_ylabel(f'{nombre}\nPA {pa:.0f}°', fontsize=8)
    print(f'{nombre:18s} PA {pa:6.1f}°  yx={[round(v,1) for v in pos]}'
          f'  mediana rojo = {np.nanmedian(fbc[rb]):9.2f}')
np.atleast_1d(axes)[-1].set_xlabel('λ [Å]')
np.atleast_1d(axes)[0].set_title('el mismo proceso donde NO hay compañero'
                                 ' (mismo radio, tres ángulos)', fontsize=9)
fig.tight_layout(); plt.show()
print()
print('compañero, para comparar   mediana rojo =',
      f'{np.nanmedian(fb[(wb >= 7500) & (wb <= 9000)]):9.2f}')


## 11 · Validación externa contra `photutils` (opcional)

La fotometría de la cadena **no usa `photutils` ni nada estilo DAOPHOT**: es una máscara de pesos propia (`musepipe/apertures.py`), binaria, y la caja se centra en el **píxel entero más cercano** al centroide de B3. Dos preguntas legítimas salen de ahí, y esta celda las contesta con una implementación independiente:

1. **¿La máscara propia es correcta?** Misma caja, mismo centro entero, `method='center'`: tiene que dar **exactamente** lo mismo. Si no, hay un bug de índices o de convención de píxel.
2. **¿Cuánto cuesta redondear el centro al píxel?** La misma caja centrada en la posición **fraccionaria** con `method='subpixel'`. Esa diferencia es una aproximación real de la cadena, no un error: mide cuánto flujo entra o sale por el desplazamiento.
3. **¿Y si la apertura fuera circular con cobertura exacta?** `CircularAperture` con `method='exact'`, cada una con **su propia** `apcorr`. Si el modelo de PSF de C1 es bueno, el flujo total debe coincidir aunque las aperturas sean distintas — es la misma lógica de la verificación box3 vs box5 de la spec.

> `photutils` **no está en `environment.yml`**: es una dependencia solo de análisis y la cadena no la necesita. Si no está, la celda lo dice y sigue.
> Para activarla: `conda install -c conda-forge photutils` en el entorno `MUSE`.


In [ ]:
try:
    from photutils.aperture import (CircularAperture, RectangularAperture,
                                    aperture_photometry)
except ImportError:
    print('photutils no está instalado: validación externa omitida.')
    print('  conda install -c conda-forge photutils   (entorno MUSE)')
else:
    paso = 20   # 1 de cada N canales: la comparación no necesita los 3681
    canales = np.arange(0, WAVE.size, paso)
    yc, xc = OBJECT_YX
    yi, xi = int(round(yc)), int(round(xc))
    size = float(APERTURE['size'])
    # photutils toma (x, y) y sitúa el centro del píxel en coordenada entera,
    # igual que los índices de numpy: las posiciones son directamente comparables.
    caja_entera = RectangularAperture([(xi, yi)], w=size, h=size, theta=0.0)
    caja_frac   = RectangularAperture([(xc, yc)], w=size, h=size, theta=0.0)
    circulo     = CircularAperture([(xc, yc)], r=size / 2.0)
    def _fot(ap, metodo, **kw):
        return np.array([float(aperture_photometry(CUBE[i], ap, method=metodo, **kw)['aperture_sum'][0])
                         for i in canales])
    pu_entera = _fot(caja_entera, 'center')
    pu_frac   = _fot(caja_frac, 'subpixel', subpixels=32)
    pu_circ   = _fot(circulo, 'exact')
    # OJO: `raw_flux` ya lleva restado el fondo de anillo; para comparar con
    # photutils, que solo suma, hay que devolvérselo.
    mio_sin_fondo = raw_flux[canales] + (bkg[canales] * npix_eff[canales]
                                        if bkg is not None else 0.0)
    # Solo canales CON dato: el hueco del láser y los bordes son NaN en el
    # cubo, y un NaN suelto convertiría el veredicto en «difieren».
    fin = np.isfinite(mio_sin_fondo) & np.isfinite(pu_entera)
    d = np.abs(mio_sin_fondo - pu_entera)[fin]
    rel = d / np.maximum(np.abs(pu_entera[fin]), 1e-30)
    igual = bool(np.allclose(mio_sin_fondo[fin], pu_entera[fin], rtol=1e-9, atol=0.0))
    print('1) máscara propia vs photutils (misma caja, centro entero, method=center)')
    print(f'   {fin.sum()} canales con dato | máx |Δ| = {d.max():.3e}'
          f'  ({100 * rel.max():.2e}%)  ->  ' + ('IDÉNTICAS' if igual else 'DIFIEREN'))
    # El efecto del centrado se mide DONDE HAY SEÑAL: en el azul el compañero
    # tiene S/N<1 y un cociente por canal solo mediría ruido.
    rojo = fin & (WAVE[canales] >= 7500) & (WAVE[canales] <= 9000)
    med_ent, med_frac = np.nanmedian(pu_entera[rojo]), np.nanmedian(pu_frac[rojo])
    print(f'2) centrar en el píxel vs en la posición real ({yc - yi:+.2f}, {xc - xi:+.2f} px),'
          f' medido en 7500–9000 Å:')
    print(f'   caja en el píxel {med_ent:10.2f} | caja en la posición real {med_frac:10.2f}'
          f' -> {100 * (med_frac - med_ent) / abs(med_ent):+.2f}%')
    dif_centro = 100 * (pu_frac - pu_entera) / np.maximum(np.abs(pu_entera), 1e-30)
    print(f'   (por canal en esa banda: mediana {np.nanmedian(dif_centro[rojo]):+.2f}%,'
          f' p5..p95 {np.nanpercentile(dif_centro[rojo], 5):+.2f}..'
          f'{np.nanpercentile(dif_centro[rojo], 95):+.2f}%)')
    # 3) flujo TOTAL: cada apertura con su propia corrección
    apc_circ, _m, _r = aperture_correction_from_psf(
        WAVE, {'kind': 'circle', 'radius_px': size / 2.0}, PSF_MODEL,
        center_yx=OBJECT_YX, correction_mode=APCORR_MODE)
    total_caja = pu_entera * apcorr[canales]
    total_circ = pu_circ * apc_circ[canales]
    razon = np.nanmedian(total_circ[rojo]) / np.nanmedian(total_caja[rojo])
    print(f'3) flujo TOTAL en 7500–9000 Å: círculo r={size / 2:.1f} px (exact) / '
          f'caja {size:.0f}×{size:.0f} = {razon:.4f}')
    print('   (1.0 = la curva de crecimiento de C1 es consistente entre aperturas;'
          ' es la misma prueba que box3 vs box5 de la spec)')
    # La curva por canal es RUIDO: se bina igual que el espectro, con su
    # error, para que se vea que la mediana de banda es lo que manda.
    paso_bin = max(1, len(canales) // 40)
    wb_c, fb_c, eb_c, _n = binea(WAVE[canales], dif_centro,
                                 np.abs(dif_centro) * 0 + np.nanstd(dif_centro), paso_bin)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
    ax1.plot(WAVE[canales], dif_centro, lw=0.5, color='0.75', label='por canal (crudo)')
    ax1.errorbar(wb_c, fb_c, yerr=eb_c, fmt='o', ms=3, lw=0.9, color='tab:purple',
                 label=f'binado {paso_bin} puntos')
    ax1.legend(fontsize=7)
    ax1.axhline(0, color='0.7', lw=0.6)
    # Umbral de interpretacion: por encima del 10% la diferencia ya no es
    # un detalle de centrado, es una discrepancia que hay que explicar.
    for lim in (-10, 10):
        ax1.axhline(lim, color='tab:red', ls=':', lw=1.1,
                    label='±10% (demasiado alto)' if lim > 0 else None)
    ax1.axvspan(7500, 9000, color='tab:red', alpha=0.07)
    ax1.set_ylim(*np.nanpercentile(dif_centro[fin], [2, 98]))
    ax1.set_ylabel('centrado [%]')
    ax1.set_title('coste de redondear el centro al píxel (banda sombreada = donde el '
                  'compañero se detecta)', fontsize=9)
    ax2.plot(WAVE[canales], total_caja, lw=0.9, label=f'caja {size:.0f}×{size:.0f} × apcorr')
    ax2.plot(WAVE[canales], total_circ, lw=0.9,
             label=f'círculo r={size / 2:.1f} px (exact) × su apcorr')
    ax2.axvspan(7500, 9000, color='tab:red', alpha=0.07)
    ax2.set_xlabel('λ [Å]'); ax2.set_ylabel('flujo total'); ax2.legend(fontsize=8)
    fig.tight_layout(); plt.show()


## 12 · Comparación con la cadena

Contra `spec_aperture_object.fits`, el producto que escribió la etapa. **Con las perillas por defecto debe salir idéntico** (a precisión de coma flotante): si no lo es, o la copia se desvió o alguna entrada no es la que usó la cadena. En cuanto cambias una perilla, esta celda mide exactamente qué se movió.


In [ ]:
from musepipe.extraction.product import SpectrumProduct
from musepipe.spectral import median_filter_1d

cadena = SpectrumProduct.read(SD / 'spec_aperture_object.fits')
ref_flux = np.asarray(cadena.flux, dtype=float)
ref_err  = np.asarray(cadena.flux_err, dtype=float)

def _compara(nombre, mio, suyo, rtol=1e-9):
    finito = np.isfinite(mio) & np.isfinite(suyo)
    dif = np.abs(mio - suyo)[finito]
    escala = np.maximum(np.abs(suyo)[finito], 1e-30)
    iguales = np.isclose(mio[finito], suyo[finito], rtol=rtol, atol=0.0)
    print(f'  {nombre:14s} idénticos {100 * iguales.mean():6.2f}% de {finito.sum()} canales'
          f' | máx |Δ| = {dif.max():.3e} ({100 * (dif / escala).max():.2e}%)')
    return bool(iguales.all())

print('mi resultado vs la cadena:')
ok = _compara('flujo', flux, ref_flux)
ok &= _compara('error', flux_err, ref_err)
ok &= _compara('apcorr', apcorr, np.asarray(cadena.apcorr, dtype=float))
print()
print('IDÉNTICO: la copia reproduce la cadena.' if ok else
      'DIFIERE — si has tocado una perilla, es lo esperado; si no, revisa el chequeo de deriva.')

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True,
                             gridspec_kw={'height_ratios': [2, 1]})
a1.plot(WAVE, median_filter_1d(ref_flux, 41), lw=1.6, color='0.6', label='cadena')
a1.plot(WAVE, median_filter_1d(flux, 41), lw=1.0, color='tab:blue', ls='--', label='este notebook')
a1.legend(fontsize=8); a1.set_ylabel('flujo (mediana 41 ch)')
a2.plot(WAVE, flux - ref_flux, lw=0.7, color='tab:purple')
a2.axhline(0, color='0.7', lw=0.6)
a2.set_ylabel('este − cadena'); a2.set_xlabel('λ [Å]')
a1.set_title('comparación con el producto de la cadena', fontsize=9)
fig.tight_layout(); plt.show()


## 13 · Y contra el QC

Los números que la etapa publicó en `spec_aperture_qc.json`, al lado de los de aquí. Sirve para ver si un cambio de perilla mueve algo que después mira D1 o E1.


In [ ]:
qc = json.loads((SD / 'spec_aperture_qc.json').read_text(encoding='utf-8'))
# El QC no publica el número de controles: viene del .npz que escribe la etapa.
ctrl_npz = SD / 'spec_aperture_controls.npz'
n_ctrl_cadena = int(np.load(ctrl_npz)['control_spectra'].shape[0]) if ctrl_npz.exists() else None
err_qc = qc.get('errors') or {}
mios = {
    'aperture_correction.median': float(np.nanmedian(apcorr)),
    'errors.mode': error_mode,
    'errors.stat_factor_box3': STAT_FACTOR,
    'errors.covariance_factor_box3': COV_FACTOR,
    'n_controles': len(controls_yx),
    'flujo mediano': float(np.nanmedian(flux)),
    'canales marcados': int((flags != 0).sum()),
}
suyos = {
    'aperture_correction.median': (qc.get('aperture_correction') or {}).get('median'),
    'errors.mode': err_qc.get('mode'),
    'errors.stat_factor_box3': err_qc.get('stat_factor_box3'),
    'errors.covariance_factor_box3': err_qc.get('covariance_factor_box3'),
    'n_controles': n_ctrl_cadena,
    'flujo mediano': float(np.nanmedian(ref_flux)),
    'canales marcados': (qc.get('flags') or {}).get('n_flagged'),
}
def _fmt(x):
    return f'{x:20.4f}' if isinstance(x, float) else f'{str(x):>20s}'
print(f"{'clave':32s} {'este notebook':>20s} {'cadena':>20s}")
for k, v in mios.items():
    w = suyos.get(k)
    marca = '' if (w is None or (isinstance(v, float) and isinstance(w, (int, float))
                                 and np.isclose(v, w, rtol=1e-9))
                   or v == w) else '   <-- difiere'
    print(f'{k:32s} {_fmt(v)} {_fmt(w)}{marca}')


## 14 · Por qué el continuo se va a negativo en el azul

En algunos objetos el continuo del extremo azul queda **por debajo de cero** en la figura anterior. La primera sospecha razonable es un error de formulación: que en vez de multiplicar por `apcorr` se estuviera restando algo. **No es eso**, y esta celda lo comprueba elemento a elemento:

> `producto = (crudo − fondo × npix_eff) × apcorr`

Cuando pasa, lo que se lee en la tabla de abajo son dos cosas encadenadas:

1. **En el azul el anillo mide más que la caja.** El fondo por píxel del anillo es *mayor* que el flujo por píxel dentro de la caja, así que `crudo − fondo × npix_eff` sale **negativo** antes de tocar la corrección de apertura. El halo AO de la primaria es **cromático** (mucho peor en el azul) y **no es plano**: a la distancia del compañero cae con el radio, así que la mediana del anillo —que abarca radios mayores y menores— no representa el fondo justo debajo de la caja.
2. **`apcorr` amplifica ese negativo.** En el azul la PSF es más ancha, la caja recoge menos luz y la corrección es varias veces la del rojo: multiplica el signo negativo por un factor grande y lo convierte en un continuo negativo llamativo.

La celda mide los tres fondos por píxel —caja, anillo y halo de la primaria a la misma distancia— así que el diagnóstico sale **para este objeto**, no heredado de otro: en unos el anillo queda por encima de la caja y el continuo azul se va a negativo, en otros no.

O sea: el signo viene del **modelo de fondo**, no de la aritmética. Es de la misma familia que el residuo de halo AO documentado para C3/psfsub, y es el motivo por el que el nivel absoluto en el azul no se cita sin la calibración de D2.


In [ ]:
BANDA_AZUL = (4800.0, 5600.0)   # cámbiala y vuelve a ejecutar

sel = (WAVE >= BANDA_AZUL[0]) & (WAVE <= BANDA_AZUL[1]) & (flags == 0)
crudo = raw_flux + (bkg * npix_eff if ANNULUS is not None else 0.0)
resta = crudo - raw_flux
med = lambda v: float(np.nanmedian(np.asarray(v, dtype=float)[sel]))
print(f'banda {BANDA_AZUL[0]:.0f}-{BANDA_AZUL[1]:.0f} Å, medianas por canal:')
print(f'  crudo (suma de la caja)      : {med(crudo):10.2f}')
print(f'  fondo × npix_eff             : {med(resta):10.2f}')
marca = '   <-- ya negativo ANTES de apcorr' if med(raw_flux) < 0 else ''
print(f'  crudo − fondo × npix_eff     : {med(raw_flux):10.2f}{marca}')
print(f'  apcorr                       : {med(apcorr):10.2f}')
print(f'  producto                     : {med(flux):10.2f}')
# La comprobación literal: no hay ninguna resta escondida en el último paso.
esperado = raw_flux * apcorr
print(f'\n¿producto == (crudo − fondo·npix) × apcorr, bit a bit?  '
      f'{np.array_equal(flux, esperado, equal_nan=True)}')

# Y de dónde sale el signo: el fondo POR PÍXEL, en tres sitios.
img = np.nanmedian(CUBE[sel], axis=0)
yy, xx = np.indices(img.shape, dtype=float)
rr_obj = np.hypot(yy - OBJECT_YX[0], xx - OBJECT_YX[1])
rr_star = np.hypot(yy - STAR_YX[0], xx - STAR_YX[1])
r_comp = float(np.hypot(OBJECT_YX[0] - STAR_YX[0], OBJECT_YX[1] - STAR_YX[1]))
en_caja = rr_obj <= 1.5
if ANNULUS is not None:
    en_anillo = ((rr_obj >= ANNULUS[0]) & (rr_obj <= ANNULUS[1])
                 & (rr_star > (ANNULUS[2] if len(ANNULUS) > 2 else 30.0)))
else:
    en_anillo = np.zeros_like(en_caja)
# El halo de la primaria a la MISMA distancia que el compañero, mirando
# alrededor: es la referencia justa, y la que el anillo no reproduce.
en_halo = (np.abs(rr_star - r_comp) <= 1.5) & (rr_obj > 6.0)
print('\nfondo por píxel en esa banda:')
for nombre, mascara in (('dentro de la caja', en_caja),
                        ('anillo del fondo', en_anillo),
                        ('halo a la misma distancia de la primaria', en_halo)):
    if mascara.any():
        print(f'  {nombre:42s}: {float(np.nanmedian(img[mascara])):7.3f}')
if med(raw_flux) < 0:
    print('  -> el anillo mide MÁS que la caja: la resta deja el continuo negativo,')
    print('     y apcorr (grande en el azul) lo amplifica. No hay error de fórmula.')
else:
    print('  -> aquí la caja queda por encima del anillo: en este objeto el continuo')
    print('     azul NO se va a negativo. El mecanismo es el mismo, el signo no.')


## Figura de paper — el espectro sin binar, con su error y sus líneas

Las figuras anteriores son de diagnóstico. Ésta es la que se publica, y por eso cambia en tres cosas:

- **Sin binar**: cada canal con su σ. Binar es cómodo para leer un continuo, pero esconde justo lo que se quiere enseñar (o no enseñar): que en Hα no hay nada por encima del ruido **a la resolución del dato**.
- **Dos barras de error**: la **empírica** (dispersión de los controles procesados igual que el objeto) como banda, y la **propagada del STAT** como línea. Que se vean las dos es la forma honesta de enseñar que el STAT del cubo no es σ ([`docs/noise_model.md`](../docs/noise_model.md)).
- **Marcado completo**: las **bandas telúricas** sombreadas por especie (O₂ naranja, H₂O cian) con la **transmisión medida esa noche** en la tira de arriba, las **líneas de acreción** por familia (Balmer, He I, prohibidas, O I, Ca II, Paschen) y las **líneas de emisión de cielo** en gris discontinuo.

### Por qué bandas telúricas y no líneas telúricas

A la resolución de MUSE (FWHM ≈ 2.5 Å) las líneas individuales de O₂ y H₂O **no se resuelven**: dentro de un píxel espectral caen muchas. Marcar líneas sueltas daría una precisión que el dato no tiene, así que se marcan **bandas**. `molecfit` no está disponible aquí y, en estos datos, **no convergió** (A3 corrigió con la estrella telúrica estándar), pero de ahí quedó una **curva de transmisión medida** en la misma rejilla de λ: eso es más específico que cualquier lista de laboratorio y es lo que se dibuja. Catálogo y curva: [`musepipe/telluric_lines.py`](../musepipe/telluric_lines.py).

### Y sus datos, en columnas

La celda **escribe la tabla** además de la figura, en **ECSV** (el estándar portable de astropy): texto plano, con las unidades y la procedencia en la cabecera, que se lee con `Table.read(ruta)` sin configurar nada y se puede mandar por correo. Una figura sin sus datos no es un resultado citable.

> **Aquí sale de TUS números**, los recalculados arriba, no del producto de la cadena: si has tocado una perilla, la figura y la tabla llevan ese cambio. Por eso se escriben con el sufijo `_debug`, en `plots/c2_aperture_debug/` y `tables/…_debug.ecsv`, y no pisan lo que exporta el notebook de auditoría.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from musepipe.paper_spectrum import (paper_spectrum_figure, pretty_flux_unit,
                                         spectrum_table_meta, write_spectrum_table)
    from musepipe.telluric_lines import measured_transmission
    ROOT_P = ROOT
    METHOD_P = 'aperture'
    PRODUCT_P = 'recalculado en C2_aperture_debug (no leído de disco)'
    TARGET_P = str(TARGET).replace(' ', '') + '_debug'
    BUNIT_P = BUNIT or 'ADU'
    W_P, F_P = WAVE, flux
    E_P, E_ALT_P = flux_err_emp, flux_err
    if not np.isfinite(E_P).any():
        E_P, E_ALT_P = flux_err, None
    EXTRA_P = {'flux_err_stat': flux_err, 'apcorr': apcorr,
               'npix_eff': npix_eff, 'flags': flags}
    MODO_P = error_mode
    # Cuando la etapa eligió el error empírico, la columna `flux_err` ES
    # la empírica: dibujar las dos encima fingiría dos estimaciones
    # independientes donde solo hay una.
    if E_ALT_P is not None and np.allclose(E_ALT_P, E_P, equal_nan=True):
        E_ALT_P = None
        EXTRA_P.pop('flux_err_stat', None)
        print('las dos columnas de error coinciden (modo empírico):'
              ' una sola banda, y una sola columna en la tabla')
    # La transmisión telúrica MEDIDA de este run (A3). Si el objeto se
    # redujo en modo `cascade` no existe suelta: se marcan las bandas del
    # catálogo sin la profundidad de esa noche, y se dice.
    trans = measured_transmission(RUN_ID, project_root=ROOT_P)
    print('transmisión telúrica:', trans['source'] if trans else
          'no medida en este run — se marcan las bandas del catálogo')
    # Los canales que la etapa marcó como malos (hueco del láser AO) no
    # se dibujan: valen 0, y un 0 pintado se lee como una medida.
    from musepipe.extraction.aperture import FLAG_BAD_WINDOW
    MALOS_P = (np.asarray(EXTRA_P.get('flags', 0), dtype=int) & FLAG_BAD_WINDOW) != 0
    fig, _ejes = paper_spectrum_figure(
        W_P, F_P, E_P, flux_err_alt=E_ALT_P, bad_channels=MALOS_P,
        err_label='±1σ empírico (controles procesados igual)', err_alt_label='±1σ propagado del STAT (no es σ)',
        transmission=trans,
        title=nb.display_name(RUN_ID) + ' · ' + 'apertura box3, rehecha en el notebook (C2 debug)',
        flux_label='flujo [' + pretty_flux_unit(BUNIT_P) + ']')
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'c2_aperture_debug'
    outdir.mkdir(parents=True, exist_ok=True)
    # PDF además de PNG: es la que va al paper, y en vectorial las
    # etiquetas de las 24 líneas siguen leyéndose al ampliar. El PNG a
    # 300 dpi es el mínimo que piden las revistas para figuras de línea.
    DPI_P = 300      # súbelo si necesitas más resolución
    for ext in ('png', 'pdf'):
        fig.savefig(outdir / ('spectrum_paper.' + ext), dpi=DPI_P)
    tabla = write_spectrum_table(
        nb.run_dir(RUN_ID) / 'tables' / ('spec_' + METHOD_P + '_' + TARGET_P + '.ecsv'),
        W_P, F_P, E_P, extra_columns=EXTRA_P,
        units={'flux': BUNIT_P, 'flux_err': BUNIT_P, 'flux_err_stat': BUNIT_P},
        meta=spectrum_table_meta(run_id=RUN_ID, target=TARGET_P, method=METHOD_P,
                                 product=PRODUCT_P, flux_unit=BUNIT_P,
                                 error_mode=MODO_P,
                                 extra={'figure': str(outdir / 'spectrum_paper.pdf')}))
    print('figura ->', outdir / 'spectrum_paper.pdf')
    print('tabla  ->', tabla, '(' + str(tabla.stat().st_size // 1024) + ' kB, '
          + str(int(np.size(W_P))) + ' canales)')
    print('        se lee con:  from astropy.table import Table; Table.read(ruta)')
    plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## 16 · Prueba — ¿y si `apcorr` fuera constante?

Una prueba, **no un cambio de la cadena**: se rehace el último paso con una corrección de apertura **plana**, `apcorr = 40` en todo el rango, en vez de la curva cromática que sale de la PSF de C1. La celda de comparación de arriba sigue midiendo contra la cadena con las perillas por defecto; esto vive aparte y escribe sus propios ficheros con sufijo `_apcorr40`.

**Qué se está suponiendo.** `apcorr = 1/f` con `f` la fracción de la PSF que cae dentro de la caja; ponerla constante equivale a decir que **esa fracción no depende de λ**. Es falso —la PSF AO se ensancha hacia el azul, así que la caja captura menos y la corrección real sube— pero es exactamente la prueba que hace falta para separar dos cosas que se confunden en la figura anterior:

- lo que en el espectro es **del compañero**, y
- lo que es **de la corrección**: una curva que va de ~118× a ~21× multiplica el azul por casi seis veces más que el rojo, y cualquier error en el modelo de PSF entra amplificado y **con pendiente**.

**Cómo leerlo.** Con `apcorr` plana el espectro es el crudo reescalado: conserva la forma medida, y la pendiente azul→rojo que se ve **es la del dato**. La diferencia entre las dos curvas es, punto por punto, lo que la corrección cromática está añadiendo a la SED. Si el continuo azul negativo se atenúa mucho al aplanar `apcorr`, confirma lo de la celda 14: el signo lo pone el fondo, pero **el tamaño lo pone la corrección**.

> Esto **no** valida `apcorr = 40` como alternativa: la corrección cromática es la física correcta y es la que usa la cadena. Es un banco de pruebas para ver cuánto del resultado depende de ella.


In [ ]:
from musepipe.paper_spectrum import (paper_spectrum_figure, pretty_flux_unit,
                                     spectrum_table_meta, write_spectrum_table)
from musepipe.telluric_lines import measured_transmission
from musepipe.spectral import median_filter_1d

APCORR_CONST = 40.0                 # la perilla de esta prueba
AZUL, ROJO = (4800.0, 5600.0), (7500.0, 9000.0)

flux_const = raw_flux * APCORR_CONST
err_const  = raw_err_emp * APCORR_CONST    # el empírico, que es el que manda

def _med(v, banda):
    sel = (WAVE >= banda[0]) & (WAVE <= banda[1]) & (flags == 0)
    return float(np.nanmedian(np.asarray(v, dtype=float)[sel]))

print(f'apcorr de la cadena : mediana {float(np.nanmedian(apcorr)):6.1f}'
      f' | azul {_med(apcorr, AZUL):6.1f} -> rojo {_med(apcorr, ROJO):6.1f}'
      f'  (razón azul/rojo = {_med(apcorr, AZUL) / _med(apcorr, ROJO):.1f}×)')
print(f'apcorr de la prueba : {APCORR_CONST:6.1f} en todo el rango (razón 1.0×)')
print()
cab = f"{'banda':22s} {'cromática':>12s} {'constante':>12s} {'const/crom':>11s}"
print(cab); print('-' * len(cab))
for nombre, banda in (('azul %.0f-%.0f Å' % AZUL, AZUL),
                      ('rojo %.0f-%.0f Å' % ROJO, ROJO)):
    a, b = _med(flux, banda), _med(flux_const, banda)
    razon = (b / a) if a not in (0.0,) and np.isfinite(a) else float('nan')
    print(f'{nombre:22s} {a:12.1f} {b:12.1f} {razon:11.2f}')
# La pendiente de la SED: cuánto sube el continuo del azul al rojo en cada caso.
for etiqueta, v in (('cromática', flux), ('constante', flux_const)):
    azul_v, rojo_v = _med(v, AZUL), _med(v, ROJO)
    if azul_v > 0:
        print(f'  pendiente rojo/azul ({etiqueta:9s}): {rojo_v / azul_v:7.2f}×')
    else:
        # Con el continuo azul negativo el cociente no es una pendiente:
        # cambia de signo y crece sin límite cerca del cero.
        print(f'  pendiente rojo/azul ({etiqueta:9s}): no se puede leer,'
              f' el continuo azul es negativo ({azul_v:.1f})')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True,
                               gridspec_kw={'height_ratios': [1, 2]})
ax1.plot(WAVE, apcorr, lw=0.9, color='tab:purple', label='apcorr de la cadena (PSF de C1)')
ax1.axhline(APCORR_CONST, color='tab:brown', ls='--', lw=1.2,
            label=f'apcorr = {APCORR_CONST:g} (prueba)')
ax1.set_ylabel('apcorr [adim.]'); ax1.legend(fontsize=8)
ax1.set_title('la corrección de apertura, y la misma puesta plana', fontsize=9)
ax2.axhline(0, color='0.6', lw=0.7)
ax2.plot(WAVE, median_filter_1d(flux, 41), lw=1.2, color='tab:blue',
         label='espectro con apcorr cromática (= la cadena)')
ax2.plot(WAVE, median_filter_1d(flux_const, 41), lw=1.2, color='tab:brown',
         label=f'espectro con apcorr = {APCORR_CONST:g}')
ax2.axvline(6562.8, color='tab:red', ls=':', lw=0.8, label='Hα')
amb = np.concatenate([median_filter_1d(flux, 41), median_filter_1d(flux_const, 41)])
ax2.set_ylim(*np.nanpercentile(amb[np.isfinite(amb)], [1, 99]))
ax2.set_xlabel('λ [Å]'); ax2.set_ylabel('flujo [' + pretty_flux_unit(BUNIT) + ']')
ax2.legend(fontsize=8)
fig.tight_layout(); plt.show()


### La misma figura de paper, con la corrección plana

Se dibuja y **se exporta** igual que la del apartado anterior, para poder ponerlas una al lado de otra. Ficheros con sufijo `_apcorr40`: no pisan nada.


In [ ]:
MALOS_P = (np.asarray(flags, dtype=int) & FLAG_BAD_WINDOW) != 0
trans = measured_transmission(RUN_ID, project_root=ROOT)
fig, _ejes = paper_spectrum_figure(
    WAVE, flux_const, err_const, bad_channels=MALOS_P, transmission=trans,
    err_label='±1σ empírico × apcorr constante',
    title=nb.display_name(RUN_ID) + ' · apertura box3 con apcorr = '
          + format(APCORR_CONST, 'g') + ' (prueba, C2 debug)',
    flux_label='flujo [' + pretty_flux_unit(BUNIT) + ']')
outdir = nb.run_dir(RUN_ID) / 'plots' / 'c2_aperture_debug'
outdir.mkdir(parents=True, exist_ok=True)
DPI_P = 300
for ext in ('png', 'pdf'):
    fig.savefig(outdir / ('spectrum_paper_apcorr40.' + ext), dpi=DPI_P)
objetivo = (nb.run_dir(RUN_ID) / 'tables'
            / ('spec_aperture_' + str(TARGET).replace(' ', '') + '_apcorr40.ecsv'))
tabla = write_spectrum_table(
    objetivo, WAVE, flux_const, err_const,
    extra_columns={'apcorr': np.full_like(WAVE, APCORR_CONST),
                   'npix_eff': npix_eff, 'flags': flags},
    units={'flux': BUNIT, 'flux_err': BUNIT},
    meta=spectrum_table_meta(
        run_id=RUN_ID, target=str(TARGET), method='aperture',
        product='prueba en C2_aperture_debug: apcorr constante',
        flux_unit=BUNIT, error_mode='empirical',
        extra={'apcorr_mode': 'constante (prueba, NO es la cadena)',
               'apcorr_value': float(APCORR_CONST),
               'figure': str(outdir / 'spectrum_paper_apcorr40.pdf')}))
print('figura ->', outdir / 'spectrum_paper_apcorr40.pdf')
print('tabla  ->', tabla)
print('   la meta dice que apcorr es constante: quien reciba el fichero suelto')
print('   no puede confundirlo con el producto de la cadena.')
plt.show()
